# 09 MLP 结构设计：输入层、隐藏层、输出层

前面我们已经讲过 MLP 为什么需要隐藏层，也讲过训练流程、优化器、正则化和 BatchNorm。

这一节开始把这些概念放回一个完整网络结构里：**一个 MLP 到底由哪些层组成，每一层在做什么，输入输出形状为什么要这样设计。**

## 1. MLP 是什么

MLP 的全称是 Multilayer Perceptron，多层感知机。

它可以先理解成由很多全连接层堆起来的神经网络。

一个最简单的 MLP 结构是：

```text
输入层 -> 隐藏层 -> 输出层
```

如果隐藏层不止一层，就是：

```text
输入层 -> 隐藏层 1 -> 隐藏层 2 -> ... -> 输出层
```

MLP 的核心特点是：每一层的每个神经元都和上一层的所有输出相连，所以也叫全连接网络。

## 2. 输入层到底是什么

输入层不是一个会计算的神经网络层。它只是数据进入模型时的形状。

如果一个样本有 $n$ 个特征，可以写成：

$$
\mathbf{x}=[x_1,x_2,\dots,x_n]
$$

那么输入维度就是：

$$
n
$$

例如预测房价时，输入特征可能包括面积、卧室数量、楼层、房龄等。如果一共有 $10$ 个特征，那么输入维度就是 $10$。

所以输入层回答的是：**每个样本用多少个数字表示。**

## 3. batch 输入的形状

训练时通常不是一次输入一个样本，而是一次输入一个 batch。

如果 batch size 是 $B$，每个样本有 $n$ 个特征，那么输入矩阵形状是：

$$
\mathbf{X}\in\mathbb{R}^{B\times n}
$$

这里：

- $B$ 表示这一批有多少个样本。
- $n$ 表示每个样本有多少个特征。

例如 batch size 是 $32$，每个样本有 $10$ 个特征：

$$
\mathbf{X}\in\mathbb{R}^{32\times 10}
$$

先记住：神经网络里的第一维通常是 batch 维。

## 4. 全连接层在做什么

MLP 的核心组件是全连接层。

一个全连接层做的是线性变换：

$$
\mathbf{Z}=\mathbf{X}\mathbf{W}+\mathbf{b}
$$

假设输入是：

$$
\mathbf{X}\in\mathbb{R}^{B\times n_{in}}
$$

输出希望有 $n_{out}$ 个神经元，那么权重矩阵形状可以理解为：

$$
\mathbf{W}\in\mathbb{R}^{n_{in}\times n_{out}}
$$

偏置形状是：

$$
\mathbf{b}\in\mathbb{R}^{n_{out}}
$$

输出形状是：

$$
\mathbf{Z}\in\mathbb{R}^{B\times n_{out}}
$$

这说明全连接层会把每个样本从 $n_{in}$ 维变换到 $n_{out}$ 维。

## 5. 隐藏层是什么

隐藏层就是输入层和输出层之间的中间层。

它之所以叫隐藏层，是因为训练数据里没有直接给出它应该是什么。训练数据只告诉我们输入 $x$ 和标签 $y$，没有告诉我们中间表示 $h$ 应该长什么样。

隐藏层的作用是学习中间表示：

$$
\mathbf{H}=\phi(\mathbf{X}\mathbf{W}+\mathbf{b})
$$

这里的 $\phi$ 是激活函数。

可以这样理解：输入层是原始特征，隐藏层是模型自己加工出来的新特征。

## 6. 为什么隐藏层后面要接激活函数

如果没有激活函数，多层线性变换叠在一起，仍然只是一个线性变换。

例如两层线性变换：

$$
\mathbf{Y}=(\mathbf{X}\mathbf{W}_1+\mathbf{b}_1)\mathbf{W}_2+\mathbf{b}_2
$$

展开后仍然可以写成：

$$
\mathbf{Y}=\mathbf{X}\mathbf{W}+\mathbf{b}
$$

这意味着没有激活函数时，堆很多层并不会真正增加非线性表达能力。

激活函数的作用就是打破这种线性限制，让模型可以表示更复杂的关系。

## 7. 隐藏层神经元数量怎么理解

如果一层隐藏层有 $k$ 个神经元，那么这一层输出就是 $k$ 维表示：

$$
\mathbf{H}\in\mathbb{R}^{B\times k}
$$

$k$ 越大，模型可以学习的中间特征越多，表达能力通常越强。

但 $k$ 不是越大越好。神经元越多，参数越多，模型越容易过拟合，计算成本也越高。

所以隐藏层宽度是在表达能力、过拟合风险和计算成本之间做选择。

## 8. 隐藏层层数怎么理解

隐藏层越多，模型可以做更多层次的特征变换。

可以粗略理解为：

```text
浅层：学习简单组合
中层：学习更抽象组合
深层：学习更复杂的层次结构
```

但层数越多，训练也越困难，可能出现梯度消失、梯度爆炸、过拟合等问题。

这就是为什么前面要讲初始化、激活函数、BatchNorm、正则化。它们都是为了让更深的网络更容易训练。

## 9. 输出层由任务决定

输出层不是随便设计的，它由任务类型决定。

常见任务有三类：

| 任务 | 例子 | 输出层含义 |
|---|---|---|
| 回归 | 预测房价 | 输出连续数值 |
| 二分类 | 判断是否患病 | 输出属于类别 $1$ 的概率或分数 |
| 多分类 | 判断数字是 0 到 9 哪一个 | 输出每个类别的分数或概率 |

隐藏层是在学习特征，输出层是在把特征变成最终答案。

## 10. 回归任务的输出层

回归任务输出的是连续数值。

例如预测房价：

$$
\hat{y}\in\mathbb{R}
$$

如果只预测一个数，输出层通常只有一个神经元：

$$
\hat{y}=\mathbf{h}\mathbf{w}+b
$$

回归输出层通常不需要 Sigmoid 或 Softmax，因为房价、温度、销量这类数值不一定限制在 $(0,1)$ 或概率分布里。

回归常用损失函数是均方误差：

$$
\mathcal{L}=\frac{1}{m}\sum_{i=1}^{m}(\hat{y}^{(i)}-y^{(i)})^2
$$

## 11. 二分类任务的输出层

二分类任务只有两个类别：

$$
y\in\{0,1\}
$$

常见做法是输出一个 logit：

$$
z=\mathbf{h}\mathbf{w}+b
$$

再通过 Sigmoid 变成概率：

$$
\hat{y}=\sigma(z)=P(y=1\mid x)
$$

如果：

$$
\hat{y}\ge 0.5
$$

就预测为类别 $1$；否则预测为类别 $0$。

二分类常用损失是二元交叉熵：

$$
\mathcal{L}=-\left[y\log(\hat{y})+(1-y)\log(1-\hat{y})\right]
$$

## 12. 多分类任务的输出层

多分类任务有 $C$ 个类别，并且每个样本只属于其中一个类别。

例如 MNIST 手写数字识别：

$$
C=10
$$

输出层通常有 $C$ 个神经元，每个神经元输出一个类别分数：

$$
\mathbf{z}=[z_1,z_2,\dots,z_C]
$$

这些分数叫 logits。

如果想得到概率，可以使用 Softmax：

$$
p_i=\frac{e^{z_i}}{\sum_{j=1}^{C}e^{z_j}}
$$

预测类别就是概率最大的类别：

$$
\hat{c}=\arg\max_i p_i
$$

多分类常用交叉熵损失。

## 13. logits 是什么

logits 是模型最后一层直接输出的原始分数。

它们还不是概率。

例如三分类模型输出：

$$
\mathbf{z}=[1.2,3.5,0.8]
$$

这表示模型认为第 $2$ 类分数最高，但这些数不满足概率要求：

$$
\sum_i z_i \ne 1
$$

而且 logit 可以是负数。

Softmax 的作用就是把 logits 转成概率分布。


## 14. 多标签分类和多分类不一样

多分类是多个类别中选一个。

多标签分类是多个标签可以同时成立。

例如一张图片里可能同时有：

```text
猫、沙发、窗户
```

这时不能用 Softmax 强行让所有类别概率加起来等于 $1$。

因为多个标签可以同时为真。

多标签任务通常对每个标签单独使用 Sigmoid：

$$
p_i=\sigma(z_i)
$$

每个标签都独立判断是否存在。

## 15. 一个 MLP 的形状例子

假设一个任务有：

- batch size 为 $32$。
- 每个样本有 $20$ 个输入特征。
- 第一隐藏层有 $64$ 个神经元。
- 第二隐藏层有 $32$ 个神经元。
- 最终做 $3$ 分类。

那么形状变化是：

$$
\mathbf{X}\in\mathbb{R}^{32\times 20}
$$

第一层：

$$
\mathbf{H}_1=\phi(\mathbf{X}\mathbf{W}_1+\mathbf{b}_1)
$$

$$
\mathbf{W}_1\in\mathbb{R}^{20\times 64},\quad \mathbf{H}_1\in\mathbb{R}^{32\times 64}
$$

第二层：

$$
\mathbf{H}_2=\phi(\mathbf{H}_1\mathbf{W}_2+\mathbf{b}_2)
$$

$$
\mathbf{W}_2\in\mathbb{R}^{64\times 32},\quad \mathbf{H}_2\in\mathbb{R}^{32\times 32}
$$

输出层：

$$
\mathbf{Z}=\mathbf{H}_2\mathbf{W}_3+\mathbf{b}_3
$$

$$
\mathbf{W}_3\in\mathbb{R}^{32\times 3},\quad \mathbf{Z}\in\mathbb{R}^{32\times 3}
$$

最后的 $3$ 表示每个样本对应 $3$ 个类别分数。

## 16. 参数量怎么数

一层全连接层的参数包括权重和偏置。

如果输入维度是 $n_{in}$，输出维度是 $n_{out}$，那么权重数量是：

$$
n_{in}\times n_{out}
$$

偏置数量是：

$$
n_{out}
$$

总参数量是：

$$
n_{in}\times n_{out}+n_{out}
$$

例如从 $20$ 维输入到 $64$ 个神经元：

$$
20\times64+64=1344
$$

参数越多，模型表达能力越强，但过拟合风险和计算成本也越高。

## 17. 如何初步设计一个 MLP

初学阶段可以按下面顺序思考：

1. 先确定输入维度。

每个样本有多少个特征？

2. 再确定任务类型。

是回归、二分类、多分类，还是多标签分类？

3. 根据任务确定输出层。

回归通常输出 $1$ 个或多个连续值；二分类通常输出 $1$ 个 logit；多分类输出 $C$ 个 logits。

4. 再设计隐藏层。

先从简单结构开始，不要一开始就堆很多层很多神经元。

5. 最后选择激活函数、初始化、正则化方法。

ReLU 家族常配 He 初始化；Tanh/Sigmoid 常配 Xavier 初始化。

## 18. 本节总结

这一节的逻辑链是：

```text
输入层决定每个样本怎么表示
-> 全连接层把输入从一个维度变换到另一个维度
-> 隐藏层学习中间特征表示
-> 激活函数提供非线性表达能力
-> 输出层由任务类型决定
-> 回归、二分类、多分类、多标签分类的输出设计不同
-> 形状和参数量必须能对上
```

先记住三个形状关系：

$$
\mathbf{X}\in\mathbb{R}^{B\times n_{in}}
$$

$$
\mathbf{W}\in\mathbb{R}^{n_{in}\times n_{out}}
$$

$$
\mathbf{X}\mathbf{W}\in\mathbb{R}^{B\times n_{out}}
$$

下一节可以开始进入一个真正的小案例：用 MLP 理解 MNIST 手写数字分类，从图片如何变成向量开始讲。